# Louisiana Tax Reform Analysis (2025)

This notebook analyzes the impact of Louisiana's tax reform package.

## Baseline (Prior Law)
- Louisiana standard deduction: **disabled** (false)
- Retirement income exemption bracket 2: **$6,000**
- Flat income tax rate: **disabled** (false)

## Reform (Current Law)
- Louisiana standard deduction: **enabled** (true)
- Retirement income exemption bracket 2: **$12,000**
- Flat income tax rate: **enabled** (true)

## Metrics
We calculate:
- Budgetary impact (revenue change)
- Winners and losers (percentage of population affected)
- Overall poverty impact
- Child poverty impact

In [10]:
from policyengine_us import Microsimulation
from policyengine_core.reforms import Reform
import pandas as pd
import numpy as np

LA_DATASET = "hf://policyengine/policyengine-us-data/states/LA.h5"
YEAR = 2025

## Helper Functions

In [11]:
def calculate_poverty(sim, period=2025, child_only=False):
    """
    Calculate poverty rate and count.
    
    Args:
        sim: Microsimulation object
        period: Year to analyze
        child_only: If True, only count children under 18
    
    Returns:
        poverty_rate: Weighted poverty rate
        people_in_poverty: Weighted count
    """
    age = np.array(sim.calculate("age", period=period))
    is_in_poverty = np.array(sim.calculate("person_in_poverty", period=period))
    person_weight = np.array(sim.calculate("person_weight", period=period))
    
    if child_only:
        mask = age < 18
    else:
        mask = np.ones_like(age, dtype=bool)
    
    # Weighted poverty rate
    weighted_in_poverty = (is_in_poverty[mask] * person_weight[mask]).sum()
    weighted_total = person_weight[mask].sum()
    poverty_rate = weighted_in_poverty / weighted_total if weighted_total > 0 else 0
    
    # Weighted count of people in poverty
    people_in_poverty = weighted_in_poverty
    
    return {
        "poverty_rate": poverty_rate,
        "people_in_poverty": people_in_poverty,
        "total_people": weighted_total
    }

def calculate_winners_losers(baseline_sim, reform_sim, period=2025):
    """
    Calculate winners and losers from a reform at the person level (weighted).
    Winners: People in households with higher net income under reform.
    Losers: People in households with lower net income under reform.
    Returns weighted counts and percentages of total population.
    """
    # Get household-level income change
    baseline_income = np.array(baseline_sim.calculate("household_net_income", period=period, map_to="household"))
    reform_income = np.array(reform_sim.calculate("household_net_income", period=period, map_to="household"))
    household_weight = np.array(baseline_sim.calculate("household_weight", period=period))
    income_change = reform_income - baseline_income
    
    # Get person-level data
    household_id_person = np.array(baseline_sim.calculate("household_id", period=period, map_to="person"))
    household_id_household = np.array(baseline_sim.calculate("household_id", period=period, map_to="household"))
    person_weight = np.array(baseline_sim.calculate("person_weight", period=period))
    
    # Create mapping of household_id to income_change
    income_change_dict = dict(zip(household_id_household, income_change))
    
    # Map income change to each person
    person_income_change = np.array([income_change_dict.get(hh_id, 0) for hh_id in household_id_person])
    
    # Weighted count of people who are winners/losers (threshold of $1)
    winners_mask = person_income_change > 1
    losers_mask = person_income_change < -1
    
    people_winning = person_weight[winners_mask].sum()
    people_losing = person_weight[losers_mask].sum()
    total_people = person_weight.sum()
    
    # Calculate percentages
    pct_winners = (people_winning / total_people * 100) if total_people > 0 else 0
    pct_losers = (people_losing / total_people * 100) if total_people > 0 else 0
    
    # Average gain/loss for winning/losing households (weighted)
    winning_hh_mask = income_change > 1
    losing_hh_mask = income_change < -1
    
    if winning_hh_mask.sum() > 0:
        avg_gain = np.average(income_change[winning_hh_mask], weights=household_weight[winning_hh_mask])
    else:
        avg_gain = 0
        
    if losing_hh_mask.sum() > 0:
        avg_loss = np.average(income_change[losing_hh_mask], weights=household_weight[losing_hh_mask])
    else:
        avg_loss = 0
    
    return {
        "people_winning": people_winning,
        "people_losing": people_losing,
        "total_people": total_people,
        "pct_winners": pct_winners,
        "pct_losers": pct_losers,
        "avg_gain": avg_gain,
        "avg_loss": avg_loss
    }

def format_currency(value):
    """Format value as currency in millions or billions."""
    if abs(value) >= 1e9:
        return f"${value/1e9:.2f}B"
    return f"${value/1e6:.2f}M"

def format_percent(value):
    """Format value as percentage."""
    return f"{value*100:.2f}%"

## Define Baseline (Prior Law)

In [12]:
def create_baseline_reform():
    """
    Baseline: Prior law settings
    - Standard deduction: disabled (false)
    - Retirement exemption bracket 2: $6,000
    - Flat tax rate: disabled (false)
    """
    reform = Reform.from_dict(
        {
            "gov.states.la.tax.income.deductions.standard.applies": {
                "2025-01-01.2100-12-31": False
            },
            "gov.states.la.tax.income.exempt_income.retirement.cap[1].amount": {
                "2025-01-01.2100-12-31": 6000
            },
            "gov.states.la.tax.income.main.flat.applies": {
                "2025-01-01.2100-12-31": False
            }
        },
        country_id="us",
    )
    return reform

print("Baseline reform function defined!")

Baseline reform function defined!


## Load Simulations

In [13]:
print("Loading baseline (prior law - old LA tax settings)...")
baseline_reform = create_baseline_reform()
baseline = Microsimulation(dataset=LA_DATASET, reform=baseline_reform)
print("Baseline loaded")

print("\nLoading reform (current law - new LA tax settings)...")
reform_sim = Microsimulation(dataset=LA_DATASET)
print("Reform loaded")

print("\n" + "="*60)
print("All simulations ready!")
print("="*60)

Loading baseline (prior law - old LA tax settings)...
Baseline loaded

Loading reform (current law - new LA tax settings)...
Reform loaded

All simulations ready!


## Calculate Impacts

In [14]:
# Baseline metrics
baseline_overall_pov = calculate_poverty(baseline, child_only=False)
baseline_child_pov = calculate_poverty(baseline, child_only=True)

# Reform metrics
reform_overall_pov = calculate_poverty(reform_sim, child_only=False)
reform_child_pov = calculate_poverty(reform_sim, child_only=True)

# Revenue impact - calculated as change in state income tax
# Note: .sum() on MicroSeries already applies weights internally
baseline_tax = baseline.calculate("la_income_tax", period=YEAR, map_to="household").sum()
reform_tax = reform_sim.calculate("la_income_tax", period=YEAR, map_to="household").sum()
revenue_change = reform_tax - baseline_tax

# Net income impact (cost to state = gain to households)
baseline_hh_income = baseline.calculate("household_net_income", period=YEAR, map_to="household").sum()
reform_hh_income = reform_sim.calculate("household_net_income", period=YEAR, map_to="household").sum()
net_income_change = reform_hh_income - baseline_hh_income

# Winners and losers (at person level)
winners_losers = calculate_winners_losers(baseline, reform_sim)

print("All impacts calculated")

All impacts calculated


## Results Summary

In [15]:
print("\n" + "="*80)
print("LOUISIANA TAX REFORM IMPACTS (2025)")
print("Baseline: Prior Law | Reform: Current Law")
print("="*80)
print("\nReform components:")
print("  1. Enable Louisiana standard deduction (false -> true)")
print("  2. Double retirement income exemption bracket 2 ($6k -> $12k)")
print("  3. Enable flat income tax rate (false -> true)")

print(f"\n{'BUDGETARY IMPACT':=^80}")
print(f"Baseline LA income tax revenue:  {format_currency(baseline_tax)}")
print(f"Reform LA income tax revenue:    {format_currency(reform_tax)}")
print(f"Revenue change:                  {format_currency(revenue_change)}")
print(f"Net income gain to households:   {format_currency(net_income_change)}")

print(f"\n{'WINNERS AND LOSERS (POPULATION)':=^80}")
print(f"People gaining income:           {winners_losers['people_winning']:,.0f} ({winners_losers['pct_winners']:.2f}% of population)")
print(f"Average gain per household:      ${winners_losers['avg_gain']:,.2f}")
print(f"People losing income:            {winners_losers['people_losing']:,.0f} ({winners_losers['pct_losers']:.2f}% of population)")
print(f"Average loss per household:      ${winners_losers['avg_loss']:,.2f}")

print(f"\n{'POVERTY IMPACT - OVERALL':=^80}")
print(f"Baseline poverty rate:           {format_percent(baseline_overall_pov['poverty_rate'])}")
print(f"Reform poverty rate:             {format_percent(reform_overall_pov['poverty_rate'])}")
overall_pov_reduction = baseline_overall_pov['poverty_rate'] - reform_overall_pov['poverty_rate']
overall_pov_pct_reduction = (overall_pov_reduction / baseline_overall_pov['poverty_rate'] * 100) if baseline_overall_pov['poverty_rate'] > 0 else 0
print(f"Absolute reduction:              {format_percent(overall_pov_reduction)}")
print(f"Relative reduction:              {overall_pov_pct_reduction:.2f}%")
people_lifted = baseline_overall_pov['people_in_poverty'] - reform_overall_pov['people_in_poverty']
print(f"People lifted from poverty:      {people_lifted:,.0f}")

print(f"\n{'POVERTY IMPACT - CHILDREN':=^80}")
print(f"Baseline child poverty rate:     {format_percent(baseline_child_pov['poverty_rate'])}")
print(f"Reform child poverty rate:       {format_percent(reform_child_pov['poverty_rate'])}")
child_pov_reduction = baseline_child_pov['poverty_rate'] - reform_child_pov['poverty_rate']
child_pov_pct_reduction = (child_pov_reduction / baseline_child_pov['poverty_rate'] * 100) if baseline_child_pov['poverty_rate'] > 0 else 0
print(f"Absolute reduction:              {format_percent(child_pov_reduction)}")
print(f"Relative reduction:              {child_pov_pct_reduction:.2f}%")
children_lifted = baseline_child_pov['people_in_poverty'] - reform_child_pov['people_in_poverty']
print(f"Children lifted from poverty:    {children_lifted:,.0f}")
print("="*80)


LOUISIANA TAX REFORM IMPACTS (2025)
Baseline: Prior Law | Reform: Current Law

Reform components:
  1. Enable Louisiana standard deduction (false -> true)
  2. Double retirement income exemption bracket 2 ($6k -> $12k)
  3. Enable flat income tax rate (false -> true)

================================BUDGETARY IMPACT================================
Baseline LA income tax revenue:  $4.93B
Reform LA income tax revenue:    $3.45B
Revenue change:                  $-1.48B
Net income gain to households:   $1.46B

========================WINNERS AND LOSERS (POPULATION)=========================
People gaining income:           3,620,437 (79.60% of population)
Average gain per household:      $1,312.58
People losing income:            135 (0.00% of population)
Average loss per household:      $-112.39

============================POVERTY IMPACT - OVERALL============================
Baseline poverty rate:           25.08%
Reform poverty rate:             24.78%
Absolute reduction:              0

In [16]:
# Calculate households benefitting (weighted)
baseline_hh_income_arr = np.array(baseline.calculate("household_net_income", period=YEAR, map_to="household"))
reform_hh_income_arr = np.array(reform_sim.calculate("household_net_income", period=YEAR, map_to="household"))
household_weight = np.array(baseline.calculate("household_weight", period=YEAR))

hh_income_change = reform_hh_income_arr - baseline_hh_income_arr
hh_benefitting_mask = hh_income_change > 1  # Gained more than $1
hh_losing_mask = hh_income_change < -1  # Lost more than $1

households_benefitting = household_weight[hh_benefitting_mask].sum()
households_losing = household_weight[hh_losing_mask].sum()
total_households = household_weight.sum()
pct_households_benefitting = (households_benefitting / total_households) * 100
pct_households_losing = (households_losing / total_households) * 100

print("="*70)
print("HOUSEHOLD-LEVEL SUMMARY")
print("="*70)
print(f"Households benefitting:        {households_benefitting:,.0f} ({pct_households_benefitting:.2f}%)")
print(f"Households losing:             {households_losing:,.0f} ({pct_households_losing:.2f}%)")
print(f"Total households:              {total_households:,.0f}")
print("="*70)

HOUSEHOLD-LEVEL SUMMARY
Households benefitting:        1,112,533 (69.12%)
Households losing:             135 (0.01%)
Total households:              1,609,614


## Distributional Analysis by Income

In [17]:
# Get household income for distributional analysis
agi = np.array(baseline.calculate("adjusted_gross_income", period=YEAR, map_to="household"))
weights = np.array(baseline.calculate("household_weight", period=YEAR))
tax_change_arr = np.array(reform_sim.calculate("household_net_income", period=YEAR, map_to="household")) - \
                 np.array(baseline.calculate("household_net_income", period=YEAR, map_to="household"))

# Income bracket analysis
income_brackets = [
    (0, 25000, "$0-$25k"),
    (25000, 50000, "$25k-$50k"),
    (50000, 75000, "$50k-$75k"),
    (75000, 100000, "$75k-$100k"),
    (100000, 150000, "$100k-$150k"),
    (150000, 250000, "$150k-$250k"),
    (250000, float('inf'), "$250k+")
]

distributional_data = []
for lower, upper, label in income_brackets:
    mask = (agi >= lower) & (agi < upper)
    if weights[mask].sum() > 0:
        hh_count = weights[mask].sum()
        # Weighted total change for this bracket
        total_change = (tax_change_arr[mask] * weights[mask]).sum()
        avg_change = total_change / hh_count
        winners_in_bracket = weights[mask & (tax_change_arr > 1)].sum()
        losers_in_bracket = weights[mask & (tax_change_arr < -1)].sum()
        
        distributional_data.append({
            "Income Bracket": label,
            "Households": f"{hh_count:,.0f}",
            "Avg Income Change": f"${avg_change:,.0f}",
            "Total Change ($M)": f"${total_change/1e6:,.1f}",
            "Winners": f"{winners_in_bracket:,.0f}",
            "Losers": f"{losers_in_bracket:,.0f}"
        })

dist_df = pd.DataFrame(distributional_data)

print("\n" + "="*100)
print("DISTRIBUTIONAL ANALYSIS BY INCOME")
print("="*100)
print(dist_df.to_string(index=False))
print("="*100)


DISTRIBUTIONAL ANALYSIS BY INCOME
Income Bracket Households Avg Income Change Total Change ($M) Winners Losers
       $0-$25k    698,970               $49             $34.2 246,524    134
     $25k-$50k    264,763              $323             $85.4 263,859      0
     $50k-$75k    172,219              $463             $79.7 172,219      0
    $75k-$100k    137,709              $637             $87.8 137,709      0
   $100k-$150k    128,017              $898            $114.9 128,017      0
   $150k-$250k     88,479            $1,608            $142.3  88,479      0
        $250k+     74,540           $12,289            $916.0  74,540      0


## Export Results

In [18]:
# Create results DataFrame
results = [
    {
        "Scenario": "LA Tax Reform",
        "Description": "Standard deduction + retirement exemption + flat tax",
        "Revenue Change": format_currency(revenue_change),
        "Net Income Change": format_currency(net_income_change),
        "Overall Poverty Change (%)": f"{overall_pov_pct_reduction:.2f}%",
        "Child Poverty Change (%)": f"{child_pov_pct_reduction:.2f}%",
        "% Population Winning": f"{winners_losers['pct_winners']:.2f}%",
        "% Population Losing": f"{winners_losers['pct_losers']:.2f}%"
    }
]

df_results = pd.DataFrame(results)

print("\n" + "="*120)
print("LA TAX REFORM SUMMARY")
print("="*120)
print(df_results.to_string(index=False))
print("="*120)

# Export to CSV
df_results.to_csv("la_tax_reform_results.csv", index=False)
print("\nExported to: la_tax_reform_results.csv")


LA TAX REFORM SUMMARY
     Scenario                                          Description Revenue Change Net Income Change Overall Poverty Change (%) Child Poverty Change (%) % Population Winning % Population Losing
LA Tax Reform Standard deduction + retirement exemption + flat tax        $-1.48B            $1.46B                      1.21%                    1.61%               79.60%               0.00%

Exported to: la_tax_reform_results.csv
